The path should be set at the 'DRAFT' folder

In [2]:
cd DRAFT

/research/phd/phd2k22/cse/rudra.dhar/DRAFT


/research/phd/phd2k22/cse/rudra.dhar/miniconda3/lib/python3.10/site-packages/IPython/core/magics/osm.py:417: UserWarning: This is now an optional IPython functionality, setting dhist requires you to install the `pickleshare` library.
  self.shell.db['dhist'] = compress_dhist(dhist)[-100:]


### Setup and common functions

In [3]:
from sentence_transformers import SentenceTransformer
import json
import faiss
import numpy as np
import os
from tqdm import tqdm
import torch
import time

In [4]:
model = SentenceTransformer(
    "Qwen/Qwen3-Embedding-8B",
    cache_folder="../cache",
    model_kwargs={"attn_implementation": "flash_attention_2", "device_map": "auto", "torch_dtype": torch.float16},
    tokenizer_kwargs={"padding_side": "left"},
)

Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

In [5]:
def load_jsonl(filepath):
    """Load JSONL into dict keyed by PrimaryKey."""
    data_map = {}
    records = []
    with open(filepath, "r", encoding="utf-8") as f:
        for line in f:
            record = json.loads(line)
            data_map[record["PrimaryKey"]] = record
            records.append(record)
    return data_map, records

def query_faiss(q_text, top_k=3):
    q_emb = model.encode([q_text], convert_to_numpy=True).astype("float32")
    scores, ids = index.search(q_emb, k=top_k)

    # Build results list (PrimaryKey, cosine similarity)
    results = [
        {"PrimaryKey": int(pk), "cosine_similarity": float(score)}
        for pk, score in zip(ids[0], scores[0])
    ]
    return results


def retrieve_few_shots(records, data_map, x, y):
    """For each datapoint: anchor + 2 retrieved neighbors."""
    fewShots = []

    for rec in tqdm(records, desc="Retrieving few shots"):
        anchor_pk = rec["PrimaryKey"]
        anchor_context = rec[x]
        anchor_decision = rec[y]

        start_time = time.time()

        # Retrieve top-3
        retrieved = query_faiss(anchor_context, top_k=3)

        # Drop the anchor if present, otherwise drop the least similar
        filtered = [r for r in retrieved if r['PrimaryKey'] != anchor_pk]
        if len(filtered) > 2:
            filtered = filtered[:2]

        # Map back to full records
        neighbors = []
        for f in filtered:
            pk = f['PrimaryKey']
            if pk in data_map:
                neighbors.append({
                    "PrimaryKey": pk,
                    x: data_map[pk][x],
                    y: data_map[pk][y]
                })
            
        retrieval_time = time.time() - start_time

        # Build final structure
        fewShots.append({
            "Anchor": {
                "PrimaryKey": anchor_pk,
                x: anchor_context,
                y: anchor_decision
            },
            "Retrieved": neighbors,
            "Time": retrieval_time
        })

    return fewShots


def save_few_shots_to_jsonl(triplets, filepath):
    """Save triplets list to JSONL file."""
    with open(filepath, "w", encoding="utf-8") as f:
        for triplet in triplets:
            f.write(json.dumps(triplet, ensure_ascii=False) + "\n")

## Context Decision

#### Validation

In [6]:
input_file = "Data/ADR-data/val.jsonl"
data_map, records = load_jsonl(input_file)

# Extract embeddingField and corresponding PrimaryKey IDs
contexts = [r["Context"] for r in records]
ids = [r["PrimaryKey"] for r in records]

# Generate embeddings with a single unified progress bar
print("🔄 Generating embeddings...")
embeddings = model.encode(
    contexts,
    batch_size=1,             # handles batching automatically
    convert_to_numpy=True,
    show_progress_bar=True     # single smooth tqdm progress bar
).astype("float32")

# --- Build cosine similarity index ---
dim = embeddings.shape[1]
index = faiss.IndexIDMap(faiss.IndexFlatIP(dim))
index.add_with_ids(embeddings, np.array(ids, dtype="int16"))

🔄 Generating embeddings...


Batches:   0%|          | 0/445 [00:00<?, ?it/s]

In [7]:
# Build fewShots (from previous step)
fewShots = retrieve_few_shots(records, data_map, x="Context", y="Decision")

# Save them
output_file = "Retrieval/CDval.jsonl"
save_few_shots_to_jsonl(fewShots, output_file)

print(f"✅ Saved {len(fewShots)} triplets to {output_file}")

Retrieving few shots: 100%|█████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 445/445 [00:21<00:00, 20.83it/s]

✅ Saved 445 triplets to Retrieval/CDval.jsonl


#### Test

In [8]:
input_file = "Data/ADR-data/test.jsonl"
data_map, records = load_jsonl(input_file)

# Extract embeddingField and corresponding PrimaryKey IDs
contexts = [r["Context"] for r in records]
ids = [r["PrimaryKey"] for r in records]

# Generate embeddings with a single unified progress bar
print("🔄 Generating embeddings...")
embeddings = model.encode(
    contexts,
    batch_size=1,             # handles batching automatically
    convert_to_numpy=True,
    show_progress_bar=True     # single smooth tqdm progress bar
).astype("float32")

# --- Build cosine similarity index ---
dim = embeddings.shape[1]
index = faiss.IndexIDMap(faiss.IndexFlatIP(dim))
index.add_with_ids(embeddings, np.array(ids, dtype="int16"))

🔄 Generating embeddings...


Batches:   0%|          | 0/891 [00:00<?, ?it/s]

In [9]:
# Build fewShots (from previous step)
fewShots = retrieve_few_shots(records, data_map, x="Context", y="Decision")

# Save them
output_file = "Retrieval/CDtest.jsonl"
save_few_shots_to_jsonl(fewShots, output_file)

print(f"✅ Saved {len(fewShots)} triplets to {output_file}")

Retrieving few shots: 100%|█████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 891/891 [00:44<00:00, 20.24it/s]


✅ Saved 891 triplets to Retrieval/CDtest.jsonl


#### Train

In [10]:
input_file = "Data/ADR-data/train.jsonl"
data_map, records = load_jsonl(input_file)

# Extract embeddingField and corresponding PrimaryKey IDs
contexts = [r["Context"] for r in records]
ids = [r["PrimaryKey"] for r in records]

# Generate embeddings with a single unified progress bar
print("🔄 Generating embeddings...")
embeddings = model.encode(
    contexts,
    batch_size=1,             # handles batching automatically
    convert_to_numpy=True,
    show_progress_bar=True     # single smooth tqdm progress bar
).astype("float32")

# --- Build cosine similarity index ---
dim = embeddings.shape[1]
index = faiss.IndexIDMap(faiss.IndexFlatIP(dim))
index.add_with_ids(embeddings, np.array(ids, dtype="int16"))

🔄 Generating embeddings...


Batches:   0%|          | 0/3115 [00:00<?, ?it/s]

In [11]:
# Build fewShots (from previous step)
fewShots = retrieve_few_shots(records, data_map, x="Context", y="Decision")

# Save them
output_file = "Retrieval/CDtrain.jsonl"
save_few_shots_to_jsonl(fewShots, output_file)

print(f"✅ Saved {len(fewShots)} triplets to {output_file}")

Retrieving few shots: 100%|███████████████████████████████████████████████████████████████████████████████████████████████████████████████| 3115/3115 [02:44<00:00, 18.99it/s]


✅ Saved 3115 triplets to Retrieval/CDtrain.jsonl


## Title Body

#### Validation

In [12]:
input_file = "Data/ADR-data/val.jsonl"
data_map, records = load_jsonl(input_file)

# Extract embeddingField and corresponding PrimaryKey IDs
contexts = [r["Title"] for r in records]
ids = [r["PrimaryKey"] for r in records]

# Generate embeddings with a single unified progress bar
print("🔄 Generating embeddings...")
embeddings = model.encode(
    contexts,
    batch_size=1,             # handles batching automatically
    convert_to_numpy=True,
    show_progress_bar=True     # single smooth tqdm progress bar
).astype("float32")

# --- Build cosine similarity index ---
dim = embeddings.shape[1]
index = faiss.IndexIDMap(faiss.IndexFlatIP(dim))
index.add_with_ids(embeddings, np.array(ids, dtype="int16"))

🔄 Generating embeddings...


Batches:   0%|          | 0/445 [00:00<?, ?it/s]

In [13]:
# Build fewShots (from previous step)
fewShots = retrieve_few_shots(records, data_map, x="Title", y="Body")

# Save them
output_file = "Retrieval/TBval.jsonl"
save_few_shots_to_jsonl(fewShots, output_file)

print(f"✅ Saved {len(fewShots)} triplets to {output_file}")

Retrieving few shots: 100%|█████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 445/445 [00:16<00:00, 26.97it/s]

✅ Saved 445 triplets to Retrieval/TBval.jsonl


#### Test

In [14]:
input_file = "Data/ADR-data/test.jsonl"
data_map, records = load_jsonl(input_file)

# Extract embeddingField and corresponding PrimaryKey IDs
contexts = [r["Title"] for r in records]
ids = [r["PrimaryKey"] for r in records]

# Generate embeddings with a single unified progress bar
print("🔄 Generating embeddings...")
embeddings = model.encode(
    contexts,
    batch_size=1,             # handles batching automatically
    convert_to_numpy=True,
    show_progress_bar=True     # single smooth tqdm progress bar
).astype("float32")

# --- Build cosine similarity index ---
dim = embeddings.shape[1]
index = faiss.IndexIDMap(faiss.IndexFlatIP(dim))
index.add_with_ids(embeddings, np.array(ids, dtype="int16"))

🔄 Generating embeddings...


Batches:   0%|          | 0/891 [00:00<?, ?it/s]

In [15]:
# Build fewShots (from previous step)
fewShots = retrieve_few_shots(records, data_map, x="Title", y="Body")

# Save them
output_file = "Retrieval/TBtest.jsonl"
save_few_shots_to_jsonl(fewShots, output_file)

print(f"✅ Saved {len(fewShots)} triplets to {output_file}")

Retrieving few shots: 100%|█████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 891/891 [00:33<00:00, 26.69it/s]

✅ Saved 891 triplets to Retrieval/TBtest.jsonl


#### Train

In [16]:
input_file = "Data/ADR-data/train.jsonl"
data_map, records = load_jsonl(input_file)

# Extract embeddingField and corresponding PrimaryKey IDs
contexts = [r["Title"] for r in records]
ids = [r["PrimaryKey"] for r in records]

# Generate embeddings with a single unified progress bar
print("🔄 Generating embeddings...")
embeddings = model.encode(
    contexts,
    batch_size=1,             # handles batching automatically
    convert_to_numpy=True,
    show_progress_bar=True     # single smooth tqdm progress bar
).astype("float32")

# --- Build cosine similarity index ---
dim = embeddings.shape[1]
index = faiss.IndexIDMap(faiss.IndexFlatIP(dim))
index.add_with_ids(embeddings, np.array(ids, dtype="int16"))

🔄 Generating embeddings...


Batches:   0%|          | 0/3115 [00:00<?, ?it/s]

In [17]:
# Build fewShots (from previous step)
fewShots = retrieve_few_shots(records, data_map, x="Title", y="Body")

# Save them
output_file = "Retrieval/TBtrain.jsonl"
save_few_shots_to_jsonl(fewShots, output_file)

print(f"✅ Saved {len(fewShots)} triplets to {output_file}")

Retrieving few shots: 100%|███████████████████████████████████████████████████████████████████████████████████████████████████████████████| 3115/3115 [02:01<00:00, 25.57it/s]


✅ Saved 3115 triplets to Retrieval/TBtrain.jsonl
